# ApertureDB Video Embeddings with TwelveLabs

This notebook demonstrates video ingestion, embedding generation using TwelveLabs Marengo model, semantic search, and connecting videos to talk entities in ApertureDB.

## Setup

### Install Dependencies

Install ApertureDB SDK and TwelveLabs client library.


In [1]:
!pip install -qU aperturedb twelvelabs

### Import Libraries

Import necessary modules for ApertureDB connection, video processing, and environment configuration.

In [2]:
# from google.colab import userdata

from aperturedb.CommonLibrary import create_connector
from aperturedb.Utils import Utils
from aperturedb.ParallelLoader import ParallelLoader
from aperturedb import Connector
from dotenv import load_dotenv
import os

### Configuration

Set ApertureDB API key from environment variables.

In [ ]:
APERTUREDB_KEY=userdata.get('APERTUREDB_KEY')


### Initialize Connection

Create ApertureDB client and define helper function to execute queries.

In [8]:
client = create_connector(key=APERTUREDB_KEY)

def run(q):
    resp, blobs = client.query(q)
    client.print_last_response()
    return resp

## Database Schema Inspection

### Retrieve Current Schema

Get the complete database schema to understand existing entities, connections, and properties.

In [9]:
run([
  {
    "GetSchema": {}
  }
])

## Video Queries

### List All Videos

Find all videos in the database and retrieve their properties including metadata and frame information.

In [6]:
run([
  {
    "FindVideo": {
      "results": {
        "list": ["name", "id", "category", "_fps", "_frame_count", "_frame_height", "_frame_width", "speaker_name", "talk_title"]
      }
    }
  }
]
)

### Query Specific Video

Find a specific video by title constraint and retrieve its blob data.

In [8]:
blob = run([
  {
    "FindVideo": {
      "constraints": {
        "talk_title": ["==", "Opening Remarks"]
      },
      "blobs": True
    }
  }
]
)

### Display Video

Query and display a specific video using ApertureDB's NotebookHelpers utility.

In [40]:
from aperturedb import NotebookHelpers as nh   # helper package for image displays and other utilities

query = [{
    "FindVideo": {
      "constraints": {
        "talk_title": ["==", "Era of Multimodal AI & Reasoning"]
      },
      "blobs": True
    }
}]

response, blobs = client.query(query)

client.print_last_response()
num_videos = response[0]["FindVideo"]["returned"]
for count in range(num_videos):
    nh.display_video_mp4(blobs[count])

### Display and Save Video

Query video, display it, and save the blob to local file system.

In [ ]:
from aperturedb import NotebookHelpers as nh   # Our helper package for image displays and other utilities

query = [{
    "FindVideo": {
      "constraints": {
        "talk_title": ["==", "Era of Multimodal AI & Reasoning"]
      },
      "blobs": True
    }
}]

response, blobs = client.query(query)

client.print_last_response()
num_videos = response[0]["FindVideo"]["returned"]
for count in range(num_videos):
    nh.display_video_mp4(blobs[count])
    # Save the video blob to a file
    with open(f"/content/video_{count}.mp4", "wb") as f:
        f.write(blobs[count])

## Video Embedding Generation

### Single Video Embedding Workflow

Complete workflow to generate and store video embeddings using TwelveLabs Marengo model. Includes video retrieval, upscaling for minimum resolution requirements, embedding generation, and storage in ApertureDB descriptor set.

In [ ]:
# You may need to install ffmpeg on your system first
# On Debian/Ubuntu: sudo apt-get install ffmpeg
# On macOS (with Homebrew): brew install ffmpeg

import os
import numpy as np
from twelvelabs import TwelveLabs
from aperturedb.CommonLibrary import create_connector
from aperturedb import NotebookHelpers as nh  # for display


TL_API_KEY = userdata.get('TL_API_KEY')
DESCRIPTOR_SET = "marengo_2_7"   # 1024-d for Marengo 2.7

client = create_connector()
tl = TwelveLabs(api_key=TL_API_KEY)

# 0) Find the target video already in ApertureDB (your snippet)
query = [{
    "FindVideo": {
      "_ref": 1,
      "constraints": { "talk_title": ["==", "Era of Multimodal AI & Reasoning"] },
      "blobs": True
    }
}]
resp, blobs = client.query(query)
client.print_last_response()
assert resp[0]["FindVideo"]["returned"] > 0, "Video not found in ApertureDB."

# (Optional) preview/save
nh.display_video_mp4(blobs[0])
open("video.mp4", "wb").write(blobs[0])

# NEW STEP: Upscale the video to meet resolution requirements (e.g., to 640x360)
# This command creates 'video_upscaled.mp4' from 'video.mp4'
os.system("ffmpeg -i video.mp4 -vf scale=640:360 video_upscaled.mp4 -y")


# 1) Make sure a 1024-d DescriptorSet exists (idempotent)
resp, _ = client.query([{
    "AddDescriptorSet": {
        "name": DESCRIPTOR_SET,
        "dimensions": 1024,
}}])

# 2) Create a whole-video embedding with TwelveLabs
# Use the NEW upscaled video file
task = tl.embed.tasks.create(
    model_name="Marengo-retrieval-2.7",
    video_file=("video_upscaled.mp4", open("video_upscaled.mp4","rb")),
    video_embedding_scope=["video", "clip"]
)
tl.embed.tasks.wait_for_done(task.id)
ve = tl.embed.tasks.retrieve(task.id)


# Grab the video-level vector (float list)
video_vec = ve.video_embedding.segments[0].float_
video_vec = np.asarray(video_vec, dtype=np.float32)

# 3) Store the embedding as a Descriptor connected to the Video
# Re-find the video to get a live ref in this transaction
q_add = [
  { "FindVideo": {
      "_ref": 1,
      "unique": True,
      "constraints": { "talk_title": ["==", "Era of Multimodal AI & Reasoning"] },
      "blobs": False
  }},
  { "AddDescriptor": {
      "set": DESCRIPTOR_SET,
      "label": "full_video",
      "connect": { "ref": 1 }   # connect descriptor -> video
  }}
]
resp, _ = client.query(q_add, blobs=[video_vec.tobytes()])



## Semantic Video Search

### Search Function

Perform semantic search on video embeddings by converting text query to embedding and finding k-nearest neighbors.

In [46]:
def search(query_text, k=5):
    """
    Performs semantic search by creating a text embedding and finding the
    k-nearest neighbors in the specified ApertureDB DescriptorSet.
    """
    # Text embedding
    t = tl.embed.create(model_name="Marengo-retrieval-2.7", text=query_text)
    qvec = np.asarray(t.text_embedding.segments[0].float_, dtype=np.float32)

    # ApertureDB query to find the nearest video descriptors
    q = [
      { "FindDescriptor": {
          "_ref": 2,
          "set": DESCRIPTOR_SET,
          "k_neighbors": k,
          "metric": "L2",      # Cosine Similarity
          "distances": True
      }},
      { "FindVideo": {
          # CORRECTED: is_connected_to expects an object, not an array.
          "is_connected_to": { "ref": 2 },
          "results": { "all_properties": True },
          "blobs": False
      }}
    ]
    res, _ = client.query(q, blobs=[qvec.tobytes()])
    return res

# Example query
results = search("speaker explains multimodal reasoning", k=3)
print("Semantic search results:")
print(results)

### Display Search Results

Parse semantic search results and display matching videos with similarity scores.


In [47]:
def display_search_results(search_results):
    """
    Parses the semantic search results list and displays the corresponding videos.
    """
    find_video_response = search_results[1]
    if find_video_response.get("FindVideo", {}).get("returned", 0) == 0:
        print("No videos found in the search results.")
        return

    # 1. Extract IDs and distances using the correct key: '_uniqueid'
    video_entities = search_results[1]["FindVideo"]["entities"]
    descriptor_entities = search_results[0]["FindDescriptor"]["entities"]

    video_ids = [entity["_uniqueid"] for entity in video_entities]
    distances = [entity["_distance"] for entity in descriptor_entities]

    # 2. Query for blobs, BUT also ask for properties to map them back
    query = [{
        "FindVideo": {
            "constraints": { "_uniqueid": ["in", video_ids] }, # Use correct key
            "blobs": True,
            "results": { "all_properties": True } # FIX: Explicitly ask for properties
        }
    }]

    response, blobs = client.query(query)

    if not blobs:
        print("Could not retrieve video data for the found IDs.")
        return

    print(f"Displaying top {len(blobs)} matches:\n")

    # 3. Map blobs to IDs for correct ordering
    # FIX: This line will now work because the query returns "entities"
    retrieved_videos = response[0]["FindVideo"]["entities"]
    # FIX: Use the correct key '_uniqueid' for the map
    blob_map = {entity["_uniqueid"]: blob for entity, blob in zip(retrieved_videos, blobs)}

    for i, video_id in enumerate(video_ids):
        title = video_entities[i].get("talk_title", "N/A")
        distance = distances[i]
        blob = blob_map.get(video_id)

        if blob:
            print(f"Match {i+1}: '{title}' (Distance: {distance:.4f})")
            nh.display_video_mp4(blob)
            print("-" * 30)

# Call the function with your results
display_search_results(results)

## Batch Embedding Generation

### Initialize Clients and Configuration

Set up logging, configuration, and initialize ApertureDB and TwelveLabs clients for batch processing.

In [6]:
import os
import numpy as np
from twelvelabs import TwelveLabs
from aperturedb.CommonLibrary import create_connector
from aperturedb import NotebookHelpers as nh
import logging
import json

# --- Basic Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Global Settings & Clients ---
# (It's generally better to pass these as arguments, but for a script, this is simple)
try:
    TL_API_KEY = userdata.get('TL_API_KEY')
    DESCRIPTOR_SET = "marengo_2_7"
    MODEL_NAME = "Marengo-retrieval-2.7"
    EMBEDDING_DIMENSIONS = 1024

    client = create_connector()
    tl = TwelveLabs(api_key=TL_API_KEY)
    logging.info("ApertureDB and TwelveLabs clients initialized.")
except Exception as e:
    logging.error(f"Failed to initialize clients: {e}")
    # Handle error appropriately if clients can't be created



### Batch Embedding Function

Process all videos in the database, generate embeddings for those without one, and store them in the descriptor set. Skips videos that already have embeddings for idempotent execution.

In [48]:
def embed_all_videos_in_db():
    """
    Finds all videos in the database, generates embeddings for those that
    don't have one, and stores them in the specified DescriptorSet.
    """
    print(f"\n{'='*50}\nStarting embedding process for DescriptorSet '{DESCRIPTOR_SET}'...\n{'='*50}", flush=True)
    logging.info(f"Starting embedding process for DescriptorSet '{DESCRIPTOR_SET}'...")

    # 1. Ensure the DescriptorSet exists (this is idempotent)
    print(f"Ensuring DescriptorSet '{DESCRIPTOR_SET}' exists...", flush=True)
    logging.info(f"Ensuring DescriptorSet '{DESCRIPTOR_SET}' exists...")
    client.query([
        {
            "AddDescriptorSet": {
                "name": DESCRIPTOR_SET,
                "dimensions": EMBEDDING_DIMENSIONS,
                "metric": "L2", # Explicitly set metric to L2
            }
        }
    ])
    print(f"DescriptorSet '{DESCRIPTOR_SET}' ensured.", flush=True)

    # 2. Find all videos in the database
    print("Finding all videos in ApertureDB...", flush=True)
    logging.info("Finding all videos in ApertureDB...")
    find_videos_q = [{"FindVideo": {"results": {"all_properties": True}}}]
    res, _ = client.query(find_videos_q)
    all_videos = res[0]["FindVideo"].get("entities", [])

    if not all_videos:
        print("No videos found in the database. Exiting.", flush=True)
        logging.warning("No videos found in the database.")
        return

    print(f"Found {len(all_videos)} videos. Checking and generating embeddings...\n", flush=True)
    logging.info(f"Found {len(all_videos)} videos. Checking and generating embeddings...")

    # 3. Loop through each video and generate an embedding if needed
    for i, video_entity in enumerate(all_videos):
        video_id = video_entity["_uniqueid"]
        video_title = video_entity.get("talk_title", video_id)
        print(f"---Processing video {i+1}/{len(all_videos)}: '{video_title}' (ID: {video_id})", flush=True)

        # Check if an embedding already exists for this video in this set
        check_q = [
          {
              "FindVideo": {
                  "constraints": {"_uniqueid": ["==", video_id]},
                  "_ref": 1
              }
          },
          {
              "FindDescriptor": {
                  "set": DESCRIPTOR_SET,
                  "is_connected_to": {"ref": 1},
                  "results": {"count": True}
              }
          }
      ]
        check_res, _ = client.query(check_q)

        desc_info = check_res[1]["FindDescriptor"]
        existing = desc_info.get("count", 0)

        if existing > 0:
            print(f"Skipping '{video_title}' (ID: {video_id}) - embedding already exists.")
            continue


        logging.info(f"Processing '{video_title}' (ID: {video_id})...")

        try:
            # 4. Get video blob
            print(f"  Retrieving video blob for '{video_title}'...", flush=True)
            get_blob_q = [{"FindVideo": {"constraints": {"_uniqueid": ["==", video_id]}, "blobs": True}}]
            _, blobs = client.query(get_blob_q)
            if not blobs:
                print(f"  ERROR: Could not retrieve blob for video ID {video_id}. Skipping.", flush=True)
                logging.error(f"Could not retrieve blob for video ID {video_id}.")
                continue
            print(f"  Blob retrieved. Size: {len(blobs[0])/1024:.2f} KB", flush=True)

            # 5. Save, upscale, and prepare video file
            print(f"  Saving and upscaling video...", flush=True)
            with open("temp_video.mp4", "wb") as f:
                f.write(blobs[0])
            os.system("ffmpeg -i temp_video.mp4 -vf scale=640:360 video_upscaled.mp4 -y -hide_banner -loglevel error") # Changed temp_video_upscaled.mp4 to video_upscaled.mp4 to match the task.create call below
            print(f"  Video upscaled.", flush=True)

            # 6. Generate embedding with TwelveLabs
            print(f"  Generating embedding with TwelveLabs for '{video_title}'...", flush=True)
            task = tl.embed.tasks.create(
                model_name=MODEL_NAME,
                video_file=("video_upscaled.mp4", open("video_upscaled.mp4", "rb")), # Use video_upscaled.mp4
                video_embedding_scope=["video", "clip"]
            )
            print(f"  TwelveLabs task created: {task.id}. Waiting for completion...", flush=True)
            tl.embed.tasks.wait_for_done(task.id, sleep_interval=5)
            ve = tl.embed.tasks.retrieve(task.id)
            video_vec = np.asarray(ve.video_embedding.segments[0].float_, dtype=np.float32)
            print(f"  Embedding generated.", flush=True)

            # 7. Store the embedding in ApertureDB
            print(f"  Storing embedding in ApertureDB for '{video_title}'...", flush=True)
            add_desc_q = [
                {"FindVideo": {"constraints": {"_uniqueid": ["==", video_id]}, "_ref": 1}},
                {"AddDescriptor": {"set": DESCRIPTOR_SET, "connect": {"ref": 1}}}
            ]
            client.query(add_desc_q, blobs=[video_vec.tobytes()])
            print(f"  Successfully created and stored embedding for '{video_title}'.", flush=True)
            logging.info(f"Successfully created and stored embedding for '{video_title}'.")

        except Exception as e:
            print(f"  ERROR: Failed to process video {video_id}: {e}", flush=True)
            logging.error(f"Failed to process video {video_id}: {e}")
        finally:
            # 8. Clean up temporary files
            if os.path.exists("temp_video.mp4"): os.remove("temp_video.mp4")
            if os.path.exists("video_upscaled.mp4"): os.remove("video_upscaled.mp4") # Remove the upscaled file

    print(f"\n{'='*50}\nEmbedding process finished.\n{'='*50}", flush=True)
    logging.info("Embedding process finished.")


### Enhanced Search and Display Function

Improved semantic search function that creates text embedding, performs k-NN search, retrieves video blobs, and displays results with similarity distances.

In [8]:
def search_and_display(query_text: str, top_n: int = 3):
    """
    Takes a text query, performs semantic search, and displays the top N matching videos.

    Args:
        query_text (str): The search query.
        top_n (int): The number of top results to retrieve and display.
    """
    logging.info(f"Performing semantic search for: '{query_text}'...")

    # 1. Create text embedding for the query
    try:
        t = tl.embed.create(model_name=MODEL_NAME, text=query_text)
        qvec = np.asarray(t.text_embedding.segments[0].float_, dtype=np.float32)
    except Exception as e:
        logging.error(f"Failed to create text embedding: {e}")
        return

    # 2. Perform the k-NN search in ApertureDB
    # IMPORTANT: give this FindDescriptor a _ref, and use that ref in is_connected_to
    search_q = [
        {
            "FindDescriptor": {
                "_ref": 1,
                "set": DESCRIPTOR_SET,
                "k_neighbors": top_n,
                "metric": "L2",      # L2 distance (smaller = more similar)
                "distances": True
            }
        },
        {
            "FindVideo": {
                "is_connected_to": {"ref": 1},
                "results": {"all_properties": True},
                "blobs": False
            }
        }
    ]

    search_results, _ = client.query(search_q, blobs=[qvec.tobytes()])
    # client.print_last_response()  # uncomment if you want to debug

    # 3. Check and parse results
    fd = search_results[0].get("FindDescriptor", {})
    fv = search_results[1].get("FindVideo", {})

    if fd.get("returned", 0) == 0:
        print("\nNo descriptors found in DescriptorSet "
              f"'{DESCRIPTOR_SET}' for this query embedding.")
        return

    if fv.get("returned", 0) == 0:
        print("\nNo videos found matching your search query.")
        return

    video_entities = fv["entities"]
    descriptor_entities = fd["entities"]

    # Make sure lengths match (they should, but let's be robust)
    n = min(len(video_entities), len(descriptor_entities))
    video_entities = video_entities[:n]
    descriptor_entities = descriptor_entities[:n]

    video_ids = [entity["_uniqueid"] for entity in video_entities]
    distances = [entity["_distance"] for entity in descriptor_entities]

    # 4. Retrieve the video blobs for display
    get_blobs_q = [{
        "FindVideo": {
            "constraints": {"_uniqueid": ["in", video_ids]},
            "blobs": True,
            "results": {"all_properties": True}
        }
    }]
    response, blobs = client.query(get_blobs_q)

    if not blobs:
        logging.error("Search returned matches, but failed to retrieve video blobs for display.")
        return

    print(f"\nDisplaying top {len(blobs)} matches for '{query_text}':\n")

    # 5. Map blobs to IDs to ensure correct order and display
    retrieved_videos = response[0]["FindVideo"]["entities"]
    blob_map = {entity["_uniqueid"]: blob for entity, blob in zip(retrieved_videos, blobs)}

    for i, video_id in enumerate(video_ids):
        title = video_entities[i].get("talk_title", "N/A")
        distance = distances[i]
        blob = blob_map.get(video_id)

        if blob:
            print(f"Match {i+1}: '{title}' (Distance: {distance:.4f})")
            nh.display_video_mp4(blob)
            print("-" * 30)


### Execute Batch Embedding

Run the batch embedding process for all videos in the database.


In [49]:
embed_all_videos_in_db()

### Test Semantic Search

Execute semantic search with example query and display top matching videos.

In [9]:
search_and_display("speaker explains multimodal reasoning", top_n=3)

# Connecting Videos to Talks

Create relationships between Video entities and Talk entities in ApertureDB using fuzzy title matching to handle minor character differences.


### Fuzzy Matching Titles 


Find the best matching Talk entity for each Video based on title similarity. Uses Python's difflib for fuzzy string matching to handle minor character differences between video and talk titles.

In [14]:
# Connecting Videos to Talks
# This section creates connections between Video and Talk entities based on fuzzy title matching

from difflib import SequenceMatcher

def calculate_similarity(str1, str2):
    """Calculate similarity ratio between two strings (0 to 1)."""
    if not str1 or not str2:
        return 0.0
    return SequenceMatcher(None, str1.lower().strip(), str2.lower().strip()).ratio()


# Configuration
SIMILARITY_THRESHOLD = 0.85
CONNECTION_CLASS = "BELONGS_TO_TALK"


# Step 1: Fetch all videos and talks
print("Fetching videos from ApertureDB...")
videos_resp = run([{
    "FindVideo": {
        "results": {"all_properties": True}
    }
}])
videos = videos_resp[0]["FindVideo"].get("entities", [])
print(f"Found {len(videos)} videos.\n")

print("Fetching talks from ApertureDB...")
talks_resp = run([{
    "FindEntity": {
        "with_class": "Talk",
        "results": {"all_properties": True}
    }
}])
talks = talks_resp[0]["FindEntity"].get("entities", [])
print(f"Found {len(talks)} talks.\n")


# Step 2: Find best matches using fuzzy matching
print(f"Matching videos to talks (threshold: {SIMILARITY_THRESHOLD})...\n")
matches = []

for video in videos:
    video_title = video.get("talk_title", "")
    video_id = video.get("_uniqueid")
    
    if not video_title:
        continue
    
    # Find best matching talk
    best_talk = None
    best_score = 0.0
    
    for talk in talks:
        talk_title = talk.get("talk_title", "")
        if not talk_title:
            continue
        
        score = calculate_similarity(video_title, talk_title)
        if score > best_score:
            best_score = score
            best_talk = talk
    
    # Add to matches if above threshold
    if best_score >= SIMILARITY_THRESHOLD and best_talk:
        matches.append({
            "video_title": video_title,
            "video_id": video_id,
            "talk_title": best_talk.get("talk_title"),
            "talk_id": best_talk.get("talk_id"),
            "similarity": best_score
        })


# Step 3: Display matches for review
print(f"\n{'='*80}")
print(f"Found {len(matches)} matches above threshold {SIMILARITY_THRESHOLD}")
print(f"{'='*80}\n")

for i, match in enumerate(matches, 1):
    print(f"{i}. Similarity: {match['similarity']:.3f}")
    print(f"   Video:  {match['video_title']}")
    print(f"   Talk:   {match['talk_title']}")
    print()

if not matches:
    print("No matches found. Try lowering the similarity threshold.")

### Create Video-Talk Connections

Create BELONGS_TO_TALK connections in ApertureDB for all matched pairs. Uses if_not_found to ensure idempotent execution (safe to re-run).

In [15]:
# Step 4: Create connections in ApertureDB
print(f"\n{'='*80}")
print(f"Creating {len(matches)} connections...")
print(f"{'='*80}\n")

success_count = 0
error_count = 0

for i, match in enumerate(matches, 1):
    try:
        # Create connection from Video to Talk
        connection_query = [
            {
                "FindVideo": {
                    "_ref": 1,
                    "constraints": {"_uniqueid": ["==", match["video_id"]]},
                    "results": {"list": ["talk_title"]}
                }
            },
            {
                "FindEntity": {
                    "_ref": 2,
                    "with_class": "Talk",
                    "constraints": {"talk_id": ["==", match["talk_id"]]},
                    "results": {"list": ["talk_title"]}
                }
            },
            {
                "AddConnection": {
                    "class": CONNECTION_CLASS,
                    "src": 1,  # Video
                    "dst": 2,  # Talk
                    "if_not_found": {}  # Only add if connection doesn't exist
                }
            }
        ]
        
        resp, _ = client.query(connection_query)
        
        if resp[2]["AddConnection"]["status"] == 0:
            print(f"{i}/{len(matches)} Connected: {match['video_title'][:60]}...")
            success_count += 1
        else:
            print(f"{i}/{len(matches)} ERROR: {match['video_title'][:60]}...")
            error_count += 1
            
    except Exception as e:
        print(f"{i}/{len(matches)} EXCEPTION: {match['video_title'][:60]}... - {e}")
        error_count += 1


print(f"Success: {success_count}, Errors: {error_count}")
print(f"{'='*80}\n")

### Verification Queries

Verify that connections were created successfully and test graph traversal from Video to Talk.

In [16]:
# Verify connections
verify_query = [{
    "FindConnection": {
        "with_class": CONNECTION_CLASS,
        "results": {"count": True}
    }
}]

resp = run(verify_query)
count = resp[0]["FindConnection"].get("count", 0)
print(f"\nTotal {CONNECTION_CLASS} connections in database: {count}")


# Example: Find talk for a specific video
example_query = [
    {
        "FindVideo": {
            "_ref": 1,
            "constraints": {"talk_title": ["==", "Era of Multimodal AI & Reasoning"]},
            "results": {"list": ["talk_title"]}
        }
    },
    {
        "FindEntity": {
            "with_class": "Talk",
            "is_connected_to": {
                "ref": 1,
                "direction": "out",
                "connection_class": CONNECTION_CLASS
            },
            "results": {"all_properties": True}
        }
    }
]

print("\nExample: Finding talk connected to a video...")
resp = run(example_query)